In [1]:
import pandas as pd
import numpy as np
import os

# Detect desktop path
if os.name == 'nt':  # Windows
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
else:  # macOS or Linux
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")

# Normalized data from Table 1
data = {
    "Alternatives": ["REF", "PG5", "PG10", "PG15", "PG20", "MW5", "MW10", "MW15", "MW20",
                     "SW5", "SW10", "SW15", "SW20", "CSW5", "CSW10", "CSW15", "CSW20"],
    "Compressive strength": [0.95, 0.93, 0.60, 0.19, 0.00, 1.00, 0.68, 0.65, 0.44,
                             0.84, 0.84, 0.69, 0.52, 0.93, 0.86, 0.70, 0.66],
    "Rate of absorption of water": [0.44, 0.58, 0.76, 0.89, 1.00, 0.23, 0.17, 0.05, 0.02,
                                    0.32, 0.07, 0.00, 0.01, 0.53, 0.51, 0.49, 0.49],
    "Open porosity": [0.46, 0.51, 0.72, 0.91, 1.00, 0.02, 0.01, 0.00, 0.06,
                      0.47, 0.48, 0.51, 0.57, 0.50, 0.55, 0.58, 0.62],
    "Electrical resistivity": [1.00, 0.95, 0.90, 0.81, 0.74, 0.95, 0.92, 0.80, 0.71,
                               0.94, 0.94, 0.70, 0.52, 0.90, 0.88, 0.38, 0.00],
    "Global Warming Potential": [1.00, 0.69, 0.39, 0.33, 0.03, 0.69, 0.39, 0.34, 0.04,
                                 0.68, 0.37, 0.31, 0.00, 0.72, 0.44, 0.42, 0.14],
    "Eco-efficiency (ISO 14045)": [0.83, 0.93, 0.64, 0.15, 0.00, 1.00, 0.74, 0.72, 0.56,
                                   0.82, 0.93, 0.78, 0.68, 0.91, 0.93, 0.75, 0.80],
    "Environmental life cycle costing": [0.00, 0.26, 0.25, 0.25, 0.25, 0.29, 0.28, 0.28, 0.27,
                                         0.14, 0.13, 0.13, 0.12, 0.55, 0.74, 0.87, 1.00],
    "Circular Economy": [0.63, 0.71, 0.80, 0.88, 0.97, 0.71, 0.80, 0.88, 0.99,
                         0.71, 0.81, 0.89, 1.00, 0.64, 0.48, 0.27, 0.00]
}

df = pd.DataFrame(data)
df.set_index("Alternatives", inplace=True)

# Weights (converted from percentage to fraction)
weights = np.array([
    21.2 / 100,   # Compressive strength
    7.9 / 100,    # Rate of absorption of water
    5.6 / 100,    # Open porosity
    10.3 / 100,   # Electrical resistivity
    18.0 / 100,   # Global Warming Potential
    17.1 / 100,   # Eco-efficiency (ISO 14045)
    12.9 / 100,   # Environmental life cycle costing
    7.2 / 100     # Circular Economy
])

# Criteria order in DataFrame
criteria = [
    "Compressive strength",
    "Rate of absorption of water",
    "Open porosity",
    "Electrical resistivity",
    "Global Warming Potential",
    "Eco-efficiency (ISO 14045)",
    "Environmental life cycle costing",
    "Circular Economy"
]

# Ensure alignment
assert list(df.columns) == criteria, "Criteria order does not match!"

# Benefit (True) or Cost (False) criteria
is_benefit = {
    "Compressive strength": True,
    "Rate of absorption of water": False,
    "Open porosity": False,
    "Electrical resistivity": True,
    "Global Warming Potential": False,
    "Eco-efficiency (ISO 14045)": True,
    "Environmental life cycle costing": True,
    "Circular Economy": True
}

# Convert cost criteria to benefit: (1 - value)
df_vikor = df.copy()
for col in df_vikor.columns:
    if not is_benefit[col]:
        df_vikor[col] = 1 - df[col]

# Apply weights
weighted_matrix = df_vikor.values * weights

# Step 1: Determine best (f*) and worst (f-) values
f_best = weighted_matrix.max(axis=0)
f_worst = weighted_matrix.min(axis=0)

# Step 2: Compute S_i (utility) and R_i (regret)
S = np.sum(weighted_matrix, axis=1)
R = np.max(weighted_matrix, axis=1)

# Step 3: Compute Q_i
S_best = S.min()
S_worst = S.max()
R_best = R.min()
R_worst = R.max()

v = 0.5  # compromise parameter

denom_S = S_worst - S_best if S_worst != S_best else 1
denom_R = R_worst - R_best if R_worst != R_best else 1

Q = v * (S - S_best) / denom_S + (1 - v) * (R - R_best) / denom_R

# Results DataFrame
results = pd.DataFrame({
    "S_i": S,
    "R_i": R,
    "Q_i": Q
}, index=df.index)

results = results.sort_values("Q_i")  # lower Q_i = better

# Output path
output_path = os.path.join(desktop, "VIKOR_Results.xlsx")

# Export to Excel
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Normalized Data")
    pd.DataFrame(weights, index=criteria, columns=["Weight"]).to_excel(writer, sheet_name="Weights")
    df_vikor.to_excel(writer, sheet_name="Adjusted Data (Benefit)")
    results.to_excel(writer, sheet_name="VIKOR Results")

print(f"✅ VIKOR file successfully saved at:\n{output_path}")


✅ VIKOR file successfully saved at:
C:\Users\Bianca\Desktop\VIKOR_Results.xlsx
